In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np

import quantem as em
from quantem.core import config
from quantem.core.datastructures import Dataset4dstem
from quantem.core.visualization import show_2d
from quantem.core.utils.diffractive_imaging_utils import fit_probe_circle

from importlib.metadata import version # Temporary fix for version check
print(f"quantEM version: {version("quantem")}")

from quantem.diffractive_imaging import (
    PtychographyDatasetRaster,
    DetectorPixelated,
    ObjectPixelated,
    ProbePixelated,
    Ptychography,
    DirectPtychography,
)


GPU_ID = 0
config.set_device(GPU_ID)
# config.set_device("mps")
print(f"device set to {config.get("device")}")


## Import and Visualize Data

In [ ]:
fdata = Path("H:/workspace/ptyrad/data/tBL_WSe2/Panel_g-h_Themis/scan_x128_y128.raw")

from ptyrad.io.generic import load_raw
data = load_raw(fdata, shape=(16384, 128, 128))
data = np.flip(data, axis=1)
data = data.reshape((128,128,128,128)).copy()
# data = data[64:,64:]
data = data.clip(0)
data = data / 151 # The convergence speed is dependent on probe / data normalization if using a fixed learning rate

In [ ]:
dset = Dataset4dstem.from_array(
        array=data,
        sampling=[0.429, 0.429, 0.0523, 0.0523],
        origin=[0,0,0,0],
        units=['A', 'A', 'A^-1', 'A^-1'],
    )

In [ ]:
dset.get_dp_mean()

show_2d(
    [
        dset[0,0].array**0.5,
        dset.dp_mean,
    ],
    titles=[
        "single DP",
        "mean DP",
    ],
);

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.imshow(data[0,0]**0.5)
plt.colorbar()
plt.show()


In [ ]:
PROBE_ENERGY = 80e3
PROBE_SEMIANGLE = 24.9 # mrad 
PROBE_DEFOCUS = 0 # A 

NUM_PROBES = 1 # can do mixed state but doesnt add anything, end up at 50/50 for 2 probe modes 
NUM_SLICES = 1 
SLICE_THICKNESS = 12 # A

In [ ]:
probe_qy0, probe_qx0, probe_R = fit_probe_circle(dset.dp_mean.array, show=True)
# dset.sampling[2] = PROBE_SEMIANGLE / probe_R
# dset.sampling[3] = PROBE_SEMIANGLE / probe_R
# dset.units[2:] = ["mrad", "mrad"]
# probe_R = PROBE_SEMIANGLE / dset.sampling[2]
# print(dset)

In [ ]:
dset.get_virtual_image(
    mode="annular",
    geometry=((probe_qy0, probe_qx0), (probe_R+10, probe_R*5)), 
    name="DF",
    show=True,
)

dset.get_virtual_image(
    mode="circle", 
    geometry=((probe_qy0, probe_qx0), probe_R+2), 
    name="BF",
    show=True,
)



In [ ]:
dset.show_virtual_images(cmap='viridis')
print(dset.shape)

## Direct Ptychography

In [ ]:
direct_ptycho = DirectPtychography.from_dataset4d(
    dset,
    energy=PROBE_ENERGY,
    semiangle_cutoff=PROBE_SEMIANGLE,
    aberration_coefs={"defocus":PROBE_DEFOCUS},
    # rotation_angle=np.deg2rad(90),
    fit_method='constant',
    max_batch_size=32*32,
    device=config.device(),
)

In [ ]:
recons = direct_ptycho._reconstruct_all_permutations(
    max_batch_size = 256*32
)

In [ ]:
em.visualization.show_2d(
    [
        recons[:3],
        recons[3:],
    ],
    title=[
        ["incoherent BF","parallax imaging","direct ptychography"],
        ["iCOM","parallax + iCOM","direct ptycho + iCOM"],
    ]
);

## Iterative Ptychography

In [ ]:
pdset = PtychographyDatasetRaster.from_dataset4dstem(
    dset,
    learn_scan_positions=True,
    learn_descan=False,
    verbose=False
)

pdset.preprocess(
    com_fit_function="constant",
    plot_rotation=True,
    plot_com=True,
    probe_energy=PROBE_ENERGY,
    force_com_rotation=False, 
    force_com_transpose=False,
    vectorized=True,
)    

### pixelated

In [ ]:
obj_model_pix = ObjectPixelated.from_uniform(
    num_slices=NUM_SLICES, 
    slice_thicknesses=SLICE_THICKNESS,
    obj_type='complex',
    device='cuda'
)

probe_params = {
    "energy" : PROBE_ENERGY,
    "defocus": PROBE_DEFOCUS,
    # "aberration_coefs": {'C10': 0},
    "semiangle_cutoff" : PROBE_SEMIANGLE, 
}

probe_model_pix = ProbePixelated.from_params(
    num_probes=NUM_PROBES,
    probe_params=probe_params,
    device='cuda'
)

detector_model = DetectorPixelated() 

ptycho_pix = Ptychography.from_models(
    dset=pdset,
    obj_model=obj_model_pix,
    probe_model=probe_model_pix,
    detector_model=detector_model,
    device='cuda',
    verbose=False, # This will disable tqmd progress bar
)

ptycho_pix.preprocess( 
    obj_padding_px=(123,123),
    batch_size=128,
)

In [ ]:
ptycho_pix.obj

In [ ]:
ptycho_pix.print_tree()

In [ ]:
opt_params = {
        "object": {
            "type": "adamw", 
            "lr": 5e-3, 
        },
        "probe": {
            "type": "adamw", 
            "lr": 5e-3, 
        },
        "dataset": { ### for optimizing over descan shifts and probe positions
            "type": "adamw",
            "lr": 1e-4,
        }
}

scheduler_params = {
    "object": { ## scheduler kwargs are passed to the scheduler (of type type)
        "type": "plateau", ## i like plateau for many cases
    },
    "probe": {
        "type": "plateau",
    },
    "dataset": { 
        "type": "plateau",  ## exp is also frequently used 
    }
}

constraints = {
    "object": {
        "tv_weight_z": 10, 
    },
    "dataset": {
        "center_scan_positions": True,
    }
}

ptycho_pix.reconstruct(
    num_iters=20,
    reset=True,
    autograd=True, 
    device=config.get("device"),
    constraints=constraints, 
    optimizer_params=opt_params,
    scheduler_params=scheduler_params,
    batch_size=128,
    store_snapshots_every=20,
).visualize(
)

In [ ]:
ptycho_pix.save("ptycho_pix.zip", mode='o')

In [ ]:
ptycho_pix.show_iters()

In [ ]:
ptycho_pix.show_obj_fft(
    cmap="magma",
    upper_quantile=0.999,
    lower_quantile=0.01,
    title="pixelated",
    axsize=(6, 6),
)


In [ ]:
opt_params = {
        "dataset": { ### goes faster if we remove the dataset optimizer
            "type": "none", ### but leave on for better recon
        },
}
ptycho_pix.reconstruct(
    optimizer_params=opt_params,
    num_iters=51,
).visualize()


In [ ]:
ptycho_pix.show_iters()

In [ ]:
ptycho_pix.print_tree()